3、工具的定义方式2：使用@tool装饰器(推荐)
使用@tool装饰器修饰，可以自动将普通 Python 函数转化为智能体可调用的工具。
此方式最直接，代码量极少，非常适合快速验证想法或创建参数简单的工具。
3.1 自定义工具描述：description
情况1：仅提供docstring信息
在bind_tools()调用时，先将函数封装为BaseTool类型的对象，再传递给convert_to_openai_tool 函
数，生成工具的描述

In [ ]:
from langchain_classic.evaluation.parsing import json_schema
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool


@tool
def get_weather(city: str):
    return f"{city}天气晴朗"


print(convert_to_openai_tool(get_weather))

### @tool会从docstring生成描述信息，同样要求遵循Google docstring 规范。如果没有docstring则报错

Traceback...
ValueError: Function must have a docstring if description not provided.

In [2]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool


@tool
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"


print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'get_weather', 'description': '天气查询工具', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}


情况2：添加工具描述：description
@tool的参数description可以更改工具描述，优先级高于docstring的函数说明

In [3]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint


@tool(description="根据城市名称查询当日天气的工具")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '根据城市名称查询当日天气的工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

情况3：解析docstring：parse_docstring
当我们没有向@tool传递description参数时，默认情况下，tool会将docstring整体视为
description，如下

In [8]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


@tool(parse_docstring=True)
def get_weather(city: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    获取当日天气，可选择是否同时查询未来五日天气预报

    Args:
        city: 城市
        units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度

        通过将parse_docstring设置为True，docstring会被解析，填充到相应的字段描述中。
        include_forecast: 是否包含未来五日的天气预报
    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温: {temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报',
        'parameters': {
            'properties': {
                'city': {'description': '城市', 'type': 'string'},
                'units': {
                    'default': 'celsius',
                    'description': '气温单位，可选：celsius-摄氏度，fahrenheit-华氏度',
                    'type': 'string'
                },
                'include_forecast': {'default': False, 'type': 'boolean'}
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

3.2 更改工具名称：name_or_callable
默认情况，使用函数名作为工具名称，但可以向@tool 传参name_or_callable，以更改工具名称。

In [9]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool


@tool(name_or_callable="getWeather")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"


print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'getWeather', 'description': '天气查询工具', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}


3.3 自定义args_schema
3.3.1 方式1：使用Pydantic模型定义

当工具的参数变得复杂，需要枚举值、范围限制或更复杂的业务逻辑验证时，Pydantic 模型是理想
的选择，提供强大的类型检查和数据验证。
使用Pydantic 的主要优势在于能够精确控制工具参数的格式和验证规则，让大模型更准确地理解如何调
用工具。
3.3.1.1 pydantic类型的定义

① BaseModel基类
通过继承核心基类BaseModel定义数据模型，从而声明字段结构、类型约束、默认值以0及校验规则。

In [14]:
from pydantic import BaseModel


class WeatherInput(BaseModel):
    city: str


print(WeatherInput(city="北京"))


@tool(description="天气查询工具", name_or_callable="getWeather", args_schema=WeatherInput)
def get_weather(city: str):
    return f"{city}天气晴朗"


print(convert_to_openai_tool(get_weather))

city='北京'
{'type': 'function', 'function': {'name': 'getWeather', 'description': '天气查询工具', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}


注意：BaseModel子类初始化时，不接收位置参数，字段值必须以关键字参数的形式传入，否则
报错。

In [11]:
from pydantic import BaseModel


class WeatherInput(BaseModel):
    city: str


print(WeatherInput("北京"))

TypeError: BaseModel.__init__() takes 1 positional argument but 2 were given

这是因为BaseModel的初始化函数签名如下
def __init__(self, /, **data: Any) -> None:

由此可知，所有关键字参数都会被收集到字典data中，然后data会按照参数类型注解进行校验，失
败时抛出异常。

② Field
Field()：用来“定制字段”的函数，可用于设置默认值、描述等。
举例1：设置默认值

In [16]:
from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    city: str = Field(
        description="城市名称",
        default="北京"
    )


print(WeatherInput(city="北京"))


@tool(description="天气查询工具", name_or_callable="getWeather", args_schema=WeatherInput)
def get_weather(city: str):
    return f"{city}天气晴朗"


rprint(convert_to_openai_tool(get_weather))

city='北京'


{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {'city': {'default': '北京', 'description': '城市名称', 'type': 'string'}},
            'type': 'object'
        }
    }
}

③ Literal
可以使用 Literal类型限定参数为固定选项。
Literal：表示字段不能是任意某种类型的值，而只能是几个固定字面量之一。

In [20]:
from typing import Literal
from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    city: str = Field(
        description="城市名称",
        default="北京"
    ),
    unit: Literal["celsius", "fahrenheit"]


print(WeatherInput(city="北京", unit="celsius"))


@tool(description="天气查询工具", name_or_callable="getWeather", args_schema=WeatherInput)
def get_weather(city: str, unit: str):
    return f"{city}天气晴朗"


rprint(convert_to_openai_tool(get_weather))

city='北京' unit='celsius'


{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {
                'city': {'type': 'string'},
                'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'}
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}

使用Json Schema定义

In [31]:
from typing import Literal
from pydantic import BaseModel, Field

json_schema = {
    'properties': {
        'city': {'type': 'string'},
        'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'}
    },
    'required': ['unit'],
    'type': 'object'
}
print(type(json_schema))
json_schema.get('properties')['include_forecast'] = {'type': 'boolean'}
rprint(json_schema)
@tool(description="天气查询工具", name_or_callable="getWeather", args_schema=json_schema)
def get_weather(city: str, unit: str, include_forecast: bool = False):
    return f"{city}天气晴朗"


rprint(convert_to_openai_tool(get_weather))

<class 'dict'>


{
    'properties': {
        'city': {'type': 'string'},
        'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
        'include_forecast': {'type': 'boolean'}
    },
    'required': ['unit'],
    'type': 'object'
}

{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {
                'city': {'type': 'string'},
                'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
                'include_forecast': {'type': 'boolean'}
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}